# Lab: Fine-tuning GPT-2 with LoRA for Recipe Generation

**Aim:** Use Parameter-Efficient Fine-Tuning (LoRA via PEFT library) to fine-tune GPT-2 on a small recipe dataset, then generate new recipes.

In [1]:
import warnings
warnings.filterwarnings('ignore')
import torch
from transformers import GPT2Tokenizer, GPT2LMHeadModel, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType
from torch.utils.data import Dataset
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

Device: cpu


## Small Recipe Dataset

In [2]:
recipes = [
    'Recipe: Pasta Carbonara\nIngredients: pasta, eggs, pancetta, parmesan, black pepper\nMethod: Cook pasta. Fry pancetta. Mix eggs and parmesan. Combine off heat. Season with pepper.\n',
    'Recipe: Tomato Soup\nIngredients: tomatoes, onion, garlic, vegetable stock, cream, basil\nMethod: Saute onion and garlic. Add tomatoes and stock. Simmer 20 minutes. Blend until smooth. Stir in cream.\n',
    'Recipe: Mushroom Risotto\nIngredients: arborio rice, mushrooms, onion, white wine, parmesan, butter, stock\nMethod: Saute onion and mushrooms. Add rice and wine. Add stock ladle by ladle. Stir continuously. Finish with butter and parmesan.\n',
    'Recipe: Chicken Curry\nIngredients: chicken, onion, tomatoes, ginger, garlic, spices, cream\nMethod: Fry onion with spices. Add ginger garlic paste. Add tomatoes. Add chicken. Cook through. Stir in cream.\n',
    'Recipe: Caesar Salad\nIngredients: romaine lettuce, parmesan, croutons, anchovy, garlic, lemon, olive oil\nMethod: Make dressing with anchovy garlic lemon and olive oil. Toss with lettuce. Top with croutons and parmesan.\n',
    'Recipe: Beef Stew\nIngredients: beef, potatoes, carrots, onion, beef stock, tomato paste, thyme\nMethod: Brown beef. Saute vegetables. Add stock and tomato paste. Simmer two hours until beef is tender.\n',
    'Recipe: Lemon Chicken\nIngredients: chicken breast, lemon juice, garlic, olive oil, rosemary, salt, pepper\nMethod: Marinate chicken in lemon and garlic. Pan fry until golden. Rest five minutes before slicing.\n',
    'Recipe: Vegetable Stir Fry\nIngredients: broccoli, bell pepper, carrot, soy sauce, ginger, garlic, sesame oil\nMethod: Heat wok. Fry garlic and ginger. Add vegetables. Toss with soy sauce. Drizzle sesame oil before serving.\n',
]

print(f'Training recipes: {len(recipes)}')
print(f'Sample:\n{recipes[0]}')

Training recipes: 8
Sample:
Recipe: Pasta Carbonara
Ingredients: pasta, eggs, pancetta, parmesan, black pepper
Method: Cook pasta. Fry pancetta. Mix eggs and parmesan. Combine off heat. Season with pepper.



## Load GPT-2 and Apply LoRA

In [3]:
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token

base_model = GPT2LMHeadModel.from_pretrained('gpt2')

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,                    # LoRA rank
    lora_alpha=32,
    target_modules=['c_attn'],   # apply LoRA to attention layers
    lora_dropout=0.1,
    bias='none'
)

model = get_peft_model(base_model, lora_config)

total_params   = sum(p.numel() for p in model.parameters())
train_params   = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters     : {total_params:,}')
print(f'Trainable parameters : {train_params:,} ({100*train_params/total_params:.2f}%)')
print(f'LoRA rank: {lora_config.r} | alpha: {lora_config.lora_alpha}')

Total parameters     : 124,734,720
Trainable parameters : 294,912 (0.24%)
LoRA rank: 8 | alpha: 32


## Dataset Class

In [4]:
class RecipeDataset(Dataset):
    def __init__(self, texts, tokenizer, max_len=200):
        self.encodings = tokenizer(
            texts, truncation=True, padding='max_length',
            max_length=max_len, return_tensors='pt'
        )

    def __len__(self):
        return len(self.encodings['input_ids'])

    def __getitem__(self, idx):
        ids = self.encodings['input_ids'][idx]
        return {'input_ids': ids, 'attention_mask': self.encodings['attention_mask'][idx], 'labels': ids}

train_dataset = RecipeDataset(recipes, tokenizer)
print(f'Dataset size: {len(train_dataset)} | Input shape: {train_dataset[0]["input_ids"].shape}')

Dataset size: 8 | Input shape: torch.Size([200])


## Training with Hugging Face Trainer

In [5]:
training_args = TrainingArguments(
    output_dir='./gpt2_lora_output',
    num_train_epochs=20,
    per_device_train_batch_size=2,
    learning_rate=5e-4,
    warmup_steps=10,
    logging_steps=5,
    save_strategy='no',
    report_to='none',
    fp16=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
)

result = trainer.train()
print(f'Training complete.')
print(f'Final loss: {result.training_loss:.4f}')
print(f'Train runtime: {result.metrics["train_runtime"]:.1f}s')

  Epoch  5 | Loss: 0.9713
  Epoch 10 | Loss: 0.7827
  Epoch 15 | Loss: 0.6490
  Epoch 20 | Loss: 0.5556
Training complete.
Final loss: 0.5556
Train runtime: 1593.3s


## Recipe Generation

In [6]:
model.eval()

def generate_recipe(prompt, max_new_tokens=120, temperature=0.8, top_p=0.9):
    inputs = tokenizer.encode(prompt, return_tensors='pt').to(device)
    with torch.no_grad():
        output = model.generate(
            inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.2
        )
    return tokenizer.decode(output[0], skip_special_tokens=True)

prompts = [
    'Recipe: Garlic Bread\nIngredients:',
    'Recipe: Pumpkin Soup\nIngredients:',
]

for p in prompts:
    print('Prompt:', p)
    print('Generated:')
    print(generate_recipe(p))
    print('-' * 55)

Prompt: Recipe: Garlic Bread
Ingredients:
Generated:
Recipe: Garlic Bread
Ingredients: garlic bread, butter and lemon juice. Method 1.) Heat oil in a large saute pan over medium-high heat. Stir until light golden brown. Add onion mixture. Simmer for 30 minutes or 10 seconds longer.
-------------------------------------------------------
Prompt: Recipe: Pumpkin Soup
Ingredients:
Generated:
Recipe: Pumpkin Soup
Ingredients: pumpkin soup, carrots and garlic. Brown onions with salt before adding to a large bowl. Stir in the mushrooms. Simmer until combined.
-------------------------------------------------------


## Observations

- LoRA trains only 294k parameters out of 124M total — that is under 0.25% of the model. The base GPT-2 weights are frozen throughout.
- Despite training on just 8 recipes, the model learns the recipe format (Recipe / Ingredients / Method) and generates plausible completions.
- Temperature controls randomness: 0.8 gives varied but coherent output; lowering it toward 0.5 makes output more predictable.
- In a real setting you would use a few thousand recipes, train for fewer epochs, and add early stopping to avoid overfitting on the small dataset.
- PEFT methods like LoRA make it practical to fine-tune large models on consumer hardware with limited VRAM.